In [ ]:
import time
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Scikit-Learn
from cuml.svm import SVC
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.utils.class_weight import compute_sample_weight, compute_class_weight
from scipy.stats import randint, uniform, loguniform

# Gradient Boosting
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

# Suppress warnings for clean output
import warnings
warnings.filterwarnings('ignore')
from sklearn.exceptions import DataConversionWarning

# Suppress the annoying LightGBM feature name warnings
warnings.filterwarnings(action='ignore', category=UserWarning, module='sklearn')
warnings.filterwarnings(action='ignore', category=UserWarning, module='lightgbm')
warnings.filterwarnings('ignore', category=UserWarning, module='xgboost')

# Helper function for Confusion Matrices
def plot_cm(y_true, y_pred, title, labels):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.title(title, fontweight='bold')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()

print("🚀 CELL 1: PREPARING DATA\n" + "="*50)

# 1. Load Data
train_data = np.load("/kaggle/input/datasets/anasibrahem101/last-split-by-me/train_data.npz")
X_train, y_train = train_data['X'], train_data['y']
train2_data = np.load("/kaggle/input/datasets/anasibrahem101/last-split-by-me/test_data.npz")
X_train2, y_train2 = train2_data['X'], train2_data['y']
X_train = np.vstack((X_train, X_train2))
y_train = np.concatenate((y_train, y_train2))

test_data = np.load("/kaggle/input/datasets/anasibrahem101/last-split-by-me/val_data.npz")
X_test, y_test = test_data['X'], test_data['y']

X_train = np.vstack((X_train, X_test))
y_train = np.concatenate((y_train, y_test))
# 2. Merge Train and Val for Cross-Validation
# X_cv = np.vstack((X_train, X_val))
# y_cv = np.concatenate((y_train, y_val))
X_cv = X_train
y_cv = y_train
# 3. Label Mapping
machine_map = {0: 0, 1: 0, 2: 1, 3: 1, 4: 2, 5: 2}
binary_map  = {0: 0, 1: 1, 2: 0, 3: 1, 4: 0, 5: 1}
target_names = ["M1_Norm", "M1_Abn", "M2_Norm", "M2_Abn", "M3_Norm", "M3_Abn"]

y_cv_mach = np.array([machine_map[y] for y in y_cv])
y_cv_bin  = np.array([binary_map[y] for y in y_cv])
y_test_mach = np.array([machine_map[y] for y in y_test])

print(f"✅ Data Ready! CV Shape: {X_cv.shape}, Test Shape: {X_test.shape}")

In [ ]:
# =============================================================================
# CELL 2 : STAGE 1 — SVM ROUTER  (fixed best params — no search)
# =============================================================================
print("🚀 CELL 2: STAGE 1 - SVM ROUTER\n" + "="*50)

X_cv_svm   = X_cv.astype(np.float64)
X_test_svm = X_test.astype(np.float64)

# ── Direct fit with known best params (skip RandomizedSearchCV) ──────────────
best_router = SVC(
    C=0.42079886696066343,
    gamma=0.0029375384576328283,
    kernel='rbf',
    class_weight='balanced',
    probability=True,
    random_state=42,
    verbose=0,
)
best_router.fit(X_cv_svm, y_cv_mach)
print("✅ Best Router Params: {'C': 0.42079886696066343, 'gamma': 0.0029375384576328283, 'kernel': 'rbf'}")

# ── Inference & timing ────────────────────────────────────────────────────────
t0 = time.perf_counter()
pred_mach_test = best_router.predict(X_test_svm)
t1 = time.perf_counter()
svm_router_time_ms = ((t1 - t0) / len(X_test_svm)) * 1000
print(f"\n⏱️ SVM Router Avg Inference Time: {svm_router_time_ms:.4f} ms / sample")

# ── Stage 1 evaluation ────────────────────────────────────────────────────────
print("\n--- Stage 1 Metrics (Machine Classification) ---")
print(classification_report(y_test_mach, pred_mach_test, target_names=["Mach 1", "Mach 2", "Mach 3"]))
plot_cm(y_test_mach, pred_mach_test, "Stage 1: SVM Machine Routing", ["Mach 1", "Mach 2", "Mach 3"])

# ── Soft probabilities for downstream ensemble ────────────────────────────────
router_probs_test = best_router.predict_proba(X_test_svm)   # shape (N, 3)

In [ ]:
# =============================================================================
# SAVE BLOCK A: SVM ROUTER
# =============================================================================
import os, pickle, json

SAVE_DIR_SVM = "saved_models/stage1_svm_router"
os.makedirs(SAVE_DIR_SVM, exist_ok=True)

# Save best SVM estimator
with open(f"{SAVE_DIR_SVM}/best_router.pkl", "wb") as f:
    pickle.dump(best_router, f)
print(f"✅ SVM Router saved → {SAVE_DIR_SVM}/best_router.pkl")

# Save metadata (hardcoded best params — no search object)
svm_meta = {
    "best_params"     : {"C": 0.42079886696066343, "gamma": 0.0029375384576328283, "kernel": "rbf"},
    "avg_inference_ms": round(svm_router_time_ms, 6),
    "input_dtype"     : "float64",
    "target"          : "machine_id",
    "target_names"    : ["Mach 1", "Mach 2", "Mach 3"],
}
with open(f"{SAVE_DIR_SVM}/metadata.json", "w") as f:
    json.dump(svm_meta, f, indent=2)
print(f"✅ Metadata saved → {SAVE_DIR_SVM}/metadata.json")

print("""
📦 saved_models/stage1_svm_router/
   ├── best_router.pkl  ← use this for inference
   └── metadata.json
""")

In [ ]:
import time
import numpy as np
from scipy.stats import uniform, randint
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import classification_report, make_scorer, f1_score

print("🚀 CELL: STAGE 2 - TUNED ISOLATION FOREST\n" + "="*50)

# 1. Custom Scorer to bridge the label mismatch
def mapped_iforest_f1(y_true, y_pred_raw):
    # IsolationForest outputs: 1 (Inlier/Normal), -1 (Outlier/Abnormal)
    # Our labels are:          0 (Normal),         1 (Abnormal)
    y_pred_mapped = np.where(y_pred_raw == 1, 0, 1)
    
    # Use macro F1 to ensure both classes are weighted equally
    return f1_score(y_true, y_pred_mapped, average='macro')

# Wrap the function into a Scikit-Learn scorer
iforest_scorer = make_scorer(mapped_iforest_f1)

# 2. Hyperparameter Space
param_dist_iforest = {
    'n_estimators': [100, 200, 300, 400],
    'max_samples': ['auto', 0.5, 0.75, 1.0],        # Subsampling ratio to build trees
    'contamination': uniform(0.01, 0.2),            # Guessing anomaly fraction between 1% and 21%
    'max_features': uniform(0.5, 0.5)               # Training trees on 50% to 100% of the 256 features
}

machine_tuned_iforests = {}

# 3. Training Loop
for mach_id in range(3):
    print(f"\n🔍 Tuning Isolation Forest for Machine {mach_id + 1}...")
    
    # Isolate data for the specific machine
    mask = (y_cv_mach == mach_id)
    X_mach, y_mach_bin = X_cv[mask], y_cv_bin[mask]
    
    iforest_base = IsolationForest(random_state=42, n_jobs=-1)
    
    # Note: Even though IForest is unsupervised, we pass 'y_mach_bin' to fit() 
    # ONLY so the RandomizedSearchCV can use it for our custom scoring function.
    random_iforest = RandomizedSearchCV(
        estimator=iforest_base,
        param_distributions=param_dist_iforest,
        n_iter=10, 
        cv=3, 
        scoring=iforest_scorer, 
        n_jobs=-1, 
        verbose=0, 
        random_state=42
    )
    
    random_iforest.fit(X_mach, y_mach_bin)
    machine_tuned_iforests[mach_id] = random_iforest.best_estimator_
    print(f"   ✅ Best Params: {random_iforest.best_params_}")

# ==========================================
# ── INFERENCE & TIMING ──
# ==========================================
t0 = time.perf_counter()
pred_mach_all = best_router.predict(X_test) # Routed by Stage 1
pred_state_iforest = np.zeros_like(pred_mach_all)

for mach_id in range(3):
    idx = np.where(pred_mach_all == mach_id)[0]
    if len(idx) > 0:
        raw_preds = machine_tuned_iforests[mach_id].predict(X_test[idx])
        # Map: 1 -> 0 (Normal), -1 -> 1 (Abnormal)
        pred_state_iforest[idx] = np.where(raw_preds == 1, 0, 1)

t1 = time.perf_counter()
iforest_final_preds = (pred_mach_all * 2) + pred_state_iforest
iforest_time_ms = ((t1 - t0) / len(X_test)) * 1000

print(f"\n⏱️ SVM + Tuned Isolation Forest Avg Inference Time: {iforest_time_ms:.4f} ms / sample")
print("\n--- Final Two-Stage Metrics (SVM + Tuned Isolation Forest) ---")
print(classification_report(y_test, iforest_final_preds, target_names=target_names))
plot_cm(y_test, iforest_final_preds, "SVM + Tuned Isolation Forest Classification", target_names)

In [ ]:
!pip install pyod
from pyod.models.hbos import HBOS
from sklearn.metrics import f1_score

print("🚀 CELL: STAGE 2 - TUNED HBOS\n" + "="*50)

param_dist_hbos = {
    'n_bins': [5, 10, 20, 50],
    'contamination': uniform(0.01, 0.2)
}

machine_tuned_hbos = {}

for mach_id in range(3):
    print(f"\n🔍 Tuning HBOS for Machine {mach_id + 1}...")
    
    mask = (y_cv_mach == mach_id)
    X_mach, y_mach_bin = X_cv[mask], y_cv_bin[mask]

    best_f1 = -1

    for bins in param_dist_hbos['n_bins']:
        for cont in [0.01, 0.05, 0.1]:
            
            hbos = HBOS(n_bins=bins, contamination=cont)
            hbos.fit(X_mach)

            preds = hbos.predict(X_mach)
            f1 = f1_score(y_mach_bin, preds, average='macro')

            if f1 > best_f1:
                best_f1 = f1
                machine_tuned_hbos[mach_id] = hbos

    print(f"   ✅ Best F1: {best_f1:.4f}")

# ==========================================
# ── INFERENCE & TIMING ──
# ==========================================
t0 = time.perf_counter()
pred_mach_all = best_router.predict(X_test)
pred_state_hbos = np.zeros_like(pred_mach_all)

for mach_id in range(3):
    idx = np.where(pred_mach_all == mach_id)[0]
    if len(idx) > 0:
        preds = machine_tuned_hbos[mach_id].predict(X_test[idx])
        pred_state_hbos[idx] = preds  # already 0/1

t1 = time.perf_counter()
hbos_final_preds = (pred_mach_all * 2) + pred_state_hbos
hbos_time_ms = ((t1 - t0) / len(X_test)) * 1000

print(f"\n⏱️ SVM + HBOS Avg Inference Time: {hbos_time_ms:.4f} ms / sample")
print("\n--- Final Two-Stage Metrics (SVM + HBOS) ---")
print(classification_report(y_test, hbos_final_preds, target_names=target_names))
plot_cm(y_test, hbos_final_preds, "SVM + HBOS Classification", target_names)

In [ ]:
# =============================================================================
# CELL 3: STAGE 2 — XGBOOST PER MACHINE (fixed best params — no search)
# =============================================================================
print("🚀 CELL 3: STAGE 2 - XGBOOST PER MACHINE\n" + "="*50)

machine_xgbs = {}

# ── Best params from prior search (same for all 3 machines) ──────────────────
best_xgb_params = {
    'colsample_bytree': 0.9795542149013333,
    'learning_rate'   : 0.26690431824362526,
    'max_depth'       : 4,
    'n_estimators'    : 364,
    'subsample'       : 0.6063865008880857,
}

for mach_id in range(3):
    print(f"\n🔧 Training XGBoost for Machine {mach_id + 1}...")
    mask = (y_cv_mach == mach_id)
    X_cv_mach, y_cv_bin_mach = X_cv[mask], y_cv_bin[mask]
    sample_weights_bin = compute_sample_weight(class_weight='balanced', y=y_cv_bin_mach)

    xgb_model = XGBClassifier(
        **best_xgb_params,
        objective='binary:logistic',
        eval_metric='logloss',
        tree_method='hist',
        device='cuda',
        feature_names=None,
        random_state=42,
    )
    xgb_model.fit(X_cv_mach, y_cv_bin_mach, sample_weight=sample_weights_bin)
    machine_xgbs[mach_id] = xgb_model
    print(f"   ✅ Machine {mach_id + 1} trained on {mask.sum()} samples")

# ── Inference & timing ────────────────────────────────────────────────────────
t0 = time.perf_counter()
pred_mach_all  = best_router.predict(X_test)
pred_state_xgb = np.zeros_like(pred_mach_all)
for mach_id in range(3):
    idx = np.where(pred_mach_all == mach_id)[0]
    if len(idx) > 0:
        pred_state_xgb[idx] = machine_xgbs[mach_id].predict(X_test[idx])
t1 = time.perf_counter()
xgb_pipeline_time_ms = ((t1 - t0) / len(X_test)) * 1000

xgb_final_preds = (pred_mach_all * 2) + pred_state_xgb

# ── Full soft probability matrix for ensemble ─────────────────────────────────
xgb_final_probs = np.zeros((len(X_test), 6))
for mach_id in range(3):
    stg2_probs = machine_xgbs[mach_id].predict_proba(X_test)
    xgb_final_probs[:, mach_id*2]     = router_probs_test[:, mach_id] * stg2_probs[:, 0]
    xgb_final_probs[:, mach_id*2 + 1] = router_probs_test[:, mach_id] * stg2_probs[:, 1]

print(f"\n⏱️ Full XGBoost Pipeline Avg Inference Time: {xgb_pipeline_time_ms:.4f} ms / sample")
print("\n--- Final Hybrid Two-Stage Metrics (SVM + 3x XGBoost) ---")
print(classification_report(y_test, xgb_final_preds, target_names=target_names))
plot_cm(y_test, xgb_final_preds, "SVM + XGBoost Classification", target_names)

In [ ]:
# =============================================================================
# SAVE BLOCK B: STAGE 2 — XGBOOST PER-MACHINE CLASSIFIERS
# =============================================================================
import os, pickle, json

SAVE_DIR_XGB = "saved_models/stage2_xgboost"
os.makedirs(SAVE_DIR_XGB, exist_ok=True)

# Save each per-machine model individually
for mach_id, xgb_model in machine_xgbs.items():
    path = f"{SAVE_DIR_XGB}/xgb_machine_{mach_id}.pkl"
    with open(path, "wb") as f:
        pickle.dump(xgb_model, f)
    print(f"✅ XGBoost Machine {mach_id + 1} saved → {path}")

# Save full dict for convenience
with open(f"{SAVE_DIR_XGB}/machine_xgbs_dict.pkl", "wb") as f:
    pickle.dump(machine_xgbs, f)
print(f"✅ Full XGBoost dict saved → {SAVE_DIR_XGB}/machine_xgbs_dict.pkl")

# Save metadata (hardcoded best params — no search object)
xgb_meta = {
    "best_params"     : best_xgb_params,
    "num_machines"    : 3,
    "objective"       : "binary:logistic",
    "avg_inference_ms": round(xgb_pipeline_time_ms, 6),
    "target_names"    : target_names,
    "note"            : "Each model predicts binary state (0/1) for its machine. Final label = (mach_id * 2) + state"
}
with open(f"{SAVE_DIR_XGB}/metadata.json", "w") as f:
    json.dump(xgb_meta, f, indent=2)
print(f"✅ Metadata saved → {SAVE_DIR_XGB}/metadata.json")

print("""
📦 saved_models/stage2_xgboost/
   ├── xgb_machine_0.pkl        ← Machine 1 binary classifier
   ├── xgb_machine_1.pkl        ← Machine 2 binary classifier
   ├── xgb_machine_2.pkl        ← Machine 3 binary classifier
   ├── machine_xgbs_dict.pkl    ← all 3 in one dict (convenience)
   └── metadata.json
""")

In [ ]:
import time
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report

print("🚀 CELL 5a: STAGE 2 - SHALLOW NEURAL NETWORK (TF)\n" + "="*50)

machine_nns = {}

def build_shallow_stage2_nn(in_dim):
    inputs = layers.Input(shape=(in_dim,))
    x = layers.Dense(128, kernel_regularizer=tf.keras.regularizers.l2(1e-3))(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.4)(x)
    
    x = layers.Dense(64, kernel_regularizer=tf.keras.regularizers.l2(1e-3))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.3)(x)
    
    outputs = layers.Dense(2, activation='softmax')(x) 
    model = models.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

for mach_id in range(3):
    print(f"\n🔍 Training Shallow NN for Machine {mach_id + 1}...")
    mask = (y_cv_mach == mach_id)
    X_mach, y_mach_bin = X_cv[mask], y_cv_bin[mask]
    
    scaler_nn = StandardScaler()
    X_mach_scaled = scaler_nn.fit_transform(X_mach)
    
    cw_bin = compute_class_weight('balanced', classes=np.unique(y_mach_bin), y=y_mach_bin)
    
    nn_model = build_shallow_stage2_nn(X_mach_scaled.shape[1])
    early_stop = callbacks.EarlyStopping(monitor='loss', patience=4, restore_best_weights=True)
    
    nn_model.fit(
        X_mach_scaled, y_mach_bin, 
        epochs=30, batch_size=128, 
        class_weight=dict(enumerate(cw_bin)), 
        callbacks=[early_stop], 
        verbose=0
    )
    machine_nns[mach_id] = (scaler_nn, nn_model)

# --- INFERENCE & EVALUATION ---
t0 = time.perf_counter()
pred_mach_all = best_router.predict(X_test) # From Stage 1
pred_state_nn = np.zeros_like(pred_mach_all)

for mach_id in range(3):
    idx = np.where(pred_mach_all == mach_id)[0]
    if len(idx) > 0:
        scaler_nn, nn_model = machine_nns[mach_id]
        X_scaled = scaler_nn.transform(X_test[idx])
        probs = nn_model.predict(X_scaled, verbose=0)
        pred_state_nn[idx] = np.argmax(probs, axis=1)

t1 = time.perf_counter()
nn_final_preds = (pred_mach_all * 2) + pred_state_nn
nn_time_ms = ((t1 - t0) / len(X_test)) * 1000

print(f"\n⏱️ SVM + Per-Machine NN Avg Inference Time: {nn_time_ms:.4f} ms / sample")
print("\n--- Final Two-Stage Metrics (SVM + NN) ---")
print(classification_report(y_test, nn_final_preds, target_names=target_names))
plot_cm(y_test, nn_final_preds, "SVM + Stage 2 NN Classification", target_names)

In [ ]:
import time
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report

print("🚀 CELL 5b: STAGE 2 - DEEP RESIDUAL MLP NN (TF)\n" + "="*50)

machine_deep_nns = {}

def build_deep_stage2_nn(in_dim):
    """
    Deeper Stage-2 NN with residual skip connections.
    Architecture: 5 dense blocks (512→256→128→64→32) with skip projections.
    """
    inputs = layers.Input(shape=(in_dim,))

    # --- Block 1: 512 ---
    x = layers.Dense(512, kernel_regularizer=tf.keras.regularizers.l2(1e-4))(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.4)(x)

    # --- Block 2: 256 + skip from block 1 (projected) ---
    skip = layers.Dense(256)(x)                           # project skip to 256
    x = layers.Dense(256, kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Add()([x, skip])

    # --- Block 3: 128 + skip ---
    skip = layers.Dense(128)(x)
    x = layers.Dense(128, kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Add()([x, skip])

    # --- Block 4: 64 + skip ---
    skip = layers.Dense(64)(x)
    x = layers.Dense(64, kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Add()([x, skip])

    # --- Block 5: 32 ---
    x = layers.Dense(32, kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.2)(x)

    outputs = layers.Dense(2, activation='softmax')(x)
    model = models.Model(inputs, outputs, name="DeepResidualMLP_Stage2")
    model.compile(
        optimizer=tf.keras.optimizers.AdamW(learning_rate=3e-4, weight_decay=1e-4),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

for mach_id in range(3):
    print(f"\n🔍 Training Deep Residual MLP for Machine {mach_id + 1}...")
    mask = (y_cv_mach == mach_id)
    X_mach, y_mach_bin = X_cv[mask], y_cv_bin[mask]

    scaler_dnn = StandardScaler()
    X_mach_scaled = scaler_dnn.fit_transform(X_mach)

    cw_bin = compute_class_weight('balanced', classes=np.unique(y_mach_bin), y=y_mach_bin)

    deep_nn = build_deep_stage2_nn(X_mach_scaled.shape[1])

    cb_list = [
        callbacks.EarlyStopping(monitor='loss', patience=6, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor='loss', factor=0.5, patience=3, min_lr=1e-6)
    ]

    deep_nn.fit(
        X_mach_scaled, y_mach_bin,
        epochs=60, batch_size=64,
        class_weight=dict(enumerate(cw_bin)),
        callbacks=cb_list,
        verbose=0
    )
    machine_deep_nns[mach_id] = (scaler_dnn, deep_nn)
    print(f"   ✅ Machine {mach_id + 1} done — {deep_nn.count_params():,} params")

# --- INFERENCE & EVALUATION ---
t0 = time.perf_counter()
pred_mach_all = best_router.predict(X_test)
pred_state_deep_nn = np.zeros_like(pred_mach_all)

deep_nn_final_probs = np.zeros((len(X_test), 6))
for mach_id in range(3):
    idx = np.where(pred_mach_all == mach_id)[0]
    if len(idx) > 0:
        scaler_dnn, deep_nn = machine_deep_nns[mach_id]
        X_scaled = scaler_dnn.transform(X_test[idx])
        probs = deep_nn.predict(X_scaled, verbose=0)
        pred_state_deep_nn[idx] = np.argmax(probs, axis=1)
        # Build full 6-class probability matrix for ensemble use
        deep_nn_final_probs[idx, mach_id*2]     = router_probs_test[idx, mach_id] * probs[:, 0]
        deep_nn_final_probs[idx, mach_id*2 + 1] = router_probs_test[idx, mach_id] * probs[:, 1]

t1 = time.perf_counter()
deep_nn_final_preds = (pred_mach_all * 2) + pred_state_deep_nn
deep_nn_time_ms = ((t1 - t0) / len(X_test)) * 1000

print(f"\n⏱️ SVM + Deep Residual MLP Avg Inference Time: {deep_nn_time_ms:.4f} ms / sample")
print("\n--- Final Two-Stage Metrics (SVM + Deep Residual MLP) ---")
print(classification_report(y_test, deep_nn_final_preds, target_names=target_names))
plot_cm(y_test, deep_nn_final_preds, "SVM + Stage 2 Deep Residual MLP", target_names)

In [ ]:
import time
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report

print("🚀 TRAINING MODEL 4: 1D ALEXNET-STYLE NEURAL NET (TENSORFLOW)\n" + "="*50)

# 1. Custom Focal Loss
class FocalLossTF(tf.keras.losses.Loss):
    def __init__(self, gamma=2.0, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
    def call(self, y_true, y_pred):
        y_true   = tf.cast(y_true, tf.int32)
        ce_loss  = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred, from_logits=False)
        pt       = tf.exp(-ce_loss)
        return tf.reduce_mean(((1.0 - pt) ** self.gamma) * ce_loss)

# 2. Model definition
def build_deep_alexnet_1d(in_dim, out_dim):
    inputs = layers.Input(shape=(in_dim,))
    x = layers.Dense(1024)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(1024)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(512)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(out_dim, activation='softmax')(x)
    return models.Model(inputs, outputs, name="DeepAlexNet1D_TF")

# 3. Scaling on ALL training data (X_train, not X_cv)
scaler_alex  = StandardScaler()
X_train_scaled = scaler_alex.fit_transform(X_train)   # ← changed from X_cv
X_test_scaled  = scaler_alex.transform(X_test)

# Class weights from full training set
cw      = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)  # ← changed from y_cv
cw_dict = dict(enumerate(cw))

# 4. Compile with Cosine Annealing
epochs         = 60
batch_size     = 32
steps_per_epoch = len(X_train_scaled) // batch_size   # ← uses full training set size

lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-3,
    decay_steps=epochs * steps_per_epoch
)

tf_alexnet = build_deep_alexnet_1d(X_train_scaled.shape[1], 6)
tf_alexnet.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=lr_schedule, weight_decay=1e-3),
    loss=FocalLossTF(gamma=2.0),
    metrics=['accuracy']
)

# 5. Training — no EarlyStopping, let all 60 epochs run on full data
print("Training 1D AlexNet (TensorFlow) on full training set...")
tf_alexnet.fit(
    X_train_scaled, y_train,          # ← changed from X_cv_scaled, y_cv
    epochs=epochs,
    batch_size=batch_size,
    class_weight=cw_dict,
    verbose=1,
    # no callbacks — EarlyStopping removed intentionally
)

# 6. Inference & timing
t0 = time.perf_counter()
probs_tf_alex = tf_alexnet.predict(X_test_scaled, batch_size=batch_size, verbose=0)
preds_tf_alex = np.argmax(probs_tf_alex, axis=1)
t1 = time.perf_counter()
tf_alex_time_ms = ((t1 - t0) / len(X_test_scaled)) * 1000

print(f"\n⏱️ Model 4 (TF AlexNet) Avg Inference Time: {tf_alex_time_ms:.4f} ms / sample")
print("\n--- MODEL 4 METRICS (1D AlexNet NN) ---")
print(classification_report(y_test, preds_tf_alex, target_names=target_names))
plot_cm(y_test, preds_tf_alex, "Model 4: 1D AlexNet Neural Net", target_names)

In [ ]:
#### SAVE ALEXNET
# =============================================================================
# SAVE BLOCK C: MODEL 4 — 1D ALEXNET TENSORFLOW
# =============================================================================
import os, pickle, json
import tensorflow as tf

SAVE_DIR_ALEX = "saved_models/model4_tf_alexnet"
os.makedirs(SAVE_DIR_ALEX, exist_ok=True)

# Save Keras model (includes architecture + weights + optimizer)
tf_alexnet.save(f"{SAVE_DIR_ALEX}/tf_alexnet.keras")
print(f"✅ Keras model saved  → {SAVE_DIR_ALEX}/tf_alexnet.keras")

# Save scaler
with open(f"{SAVE_DIR_ALEX}/scaler_alex.pkl", "wb") as f:
    pickle.dump(scaler_alex, f)
print(f"✅ Scaler saved       → {SAVE_DIR_ALEX}/scaler_alex.pkl")

# Save class weights
with open(f"{SAVE_DIR_ALEX}/class_weights.json", "w") as f:
    json.dump({str(k): v for k, v in cw_dict.items()}, f, indent=2)
print(f"✅ Class weights saved → {SAVE_DIR_ALEX}/class_weights.json")

# Save metadata
alex_meta = {
    "input_dim"      : int(X_train_scaled.shape[1]),
    "output_dim"      : 6,
    "epochs_trained"  : len(tf_alexnet.history.history["loss"]),
    "batch_size"      : batch_size,
    "avg_inference_ms": round(tf_alex_time_ms, 6),
    "target_names"    : target_names,
}
with open(f"{SAVE_DIR_ALEX}/metadata.json", "w") as f:
    json.dump(alex_meta, f, indent=2)
print(f"✅ Metadata saved     → {SAVE_DIR_ALEX}/metadata.json")

print("""
📦 saved_models/model4_tf_alexnet/
   ├── tf_alexnet.keras   ← Keras model (architecture + weights)
   ├── scaler_alex.pkl    ← StandardScaler (must use for inference)
   ├── class_weights.json
   └── metadata.json
""")

In [ ]:
import time
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report
# Add to Cell 1, after the TF imports

class FocalLossTF(tf.keras.losses.Loss):
    def __init__(self, gamma=2.0, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.int32)
        ce_loss = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred, from_logits=False)
        pt = tf.exp(-ce_loss)
        return tf.reduce_mean(((1.0 - pt) ** self.gamma) * ce_loss)
        
print("🚀 TRAINING MODEL 6: FEATURE TRANSFORMER (SELF-ATTENTION MLP)\n" + "="*50)

def build_feature_transformer(in_dim, out_dim, num_heads=4, ff_dim=256, num_blocks=3, embed_dim=64):
    """
    Transformer encoder for tabular/flat feature data.
    Each feature is projected to embed_dim and treated as a token.
    Multi-head self-attention captures global feature interactions.
    """
    inputs = layers.Input(shape=(in_dim,))

    # --- Embed each feature as a token: (batch, in_dim) → (batch, in_dim, embed_dim) ---
    x = layers.Reshape((in_dim, 1))(inputs)
    x = layers.Dense(embed_dim)(x)                          # linear projection per feature

    # --- Learnable positional encoding ---
    positions = tf.range(start=0, limit=in_dim, delta=1)
    pos_embed = layers.Embedding(input_dim=in_dim, output_dim=embed_dim)(positions)
    x = x + pos_embed                                       # broadcast add

    # --- Transformer Encoder Blocks ---
    for _ in range(num_blocks):
        # Multi-Head Self-Attention
        attn_out = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim // num_heads, dropout=0.1
        )(x, x)
        x = layers.Add()([x, attn_out])
        x = layers.LayerNormalization(epsilon=1e-6)(x)

        # Feed-Forward sublayer
        ff = layers.Dense(ff_dim, activation='relu')(x)
        ff = layers.Dropout(0.2)(ff)
        ff = layers.Dense(embed_dim)(ff)
        x = layers.Add()([x, ff])
        x = layers.LayerNormalization(epsilon=1e-6)(x)

    # --- Classification head ---
    x = layers.GlobalAveragePooling1D()(x)                 # aggregate across token dim
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(out_dim, activation='softmax')(x)

    return models.Model(inputs, outputs, name="FeatureTransformer")

# --- Prepare data ---
scaler_transformer = StandardScaler()
X_cv_tr   = scaler_transformer.fit_transform(X_cv)
X_test_tr = scaler_transformer.transform(X_test)

cw = compute_class_weight('balanced', classes=np.unique(y_cv), y=y_cv)
cw_dict = dict(enumerate(cw))

epochs = 60
batch_size = 32
steps_per_epoch = len(X_cv_tr) // batch_size

lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=5e-4, decay_steps=epochs * steps_per_epoch, alpha=1e-6
)

transformer_model = build_feature_transformer(X_cv_tr.shape[1], 6)
transformer_model.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=lr_schedule, weight_decay=1e-4),
    loss=FocalLossTF(gamma=2.0),
    metrics=['accuracy']
)
transformer_model.summary()

cb_list = [
    callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
]

print("\nTraining Feature Transformer...")
transformer_model.fit(
    X_cv_tr, y_cv,
    epochs=epochs, batch_size=batch_size,
    validation_split=0.1,
    class_weight=cw_dict,
    callbacks=cb_list,
    verbose=1
)

# --- Inference & Evaluation ---
t0 = time.perf_counter()
probs_transformer = transformer_model.predict(X_test_tr, batch_size=batch_size, verbose=0)
preds_transformer = np.argmax(probs_transformer, axis=1)
t1 = time.perf_counter()

transformer_time_ms = ((t1 - t0) / len(X_test_tr)) * 1000

print(f"\n⏱️ Model 6 (Feature Transformer) Avg Inference Time: {transformer_time_ms:.4f} ms / sample")
print("\n--- MODEL 6 METRICS (Feature Transformer) ---")
print(classification_report(y_test, preds_transformer, target_names=target_names))
plot_cm(y_test, preds_transformer, "Model 6: Feature Transformer", target_names)

In [ ]:
import scipy.stats as stats
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, accuracy_score

print("🤝 CELL 6: ENSEMBLE & CONFLICT ANALYSIS")
print("=" * 50)

# ─────────────────────────────────────────────────────
# 0. REGISTER ALL TRAINED MODELS
#    Each entry: (name, final_preds, final_probs_6class)
#    probs must be shape (N, 6) — already weighted by router where applicable
# ─────────────────────────────────────────────────────

# Build AlexNet 6-class prob matrix (end-to-end, no router weighting needed)
alexnet_final_probs = probs_tf_alex                          # shape (N, 6) directly
alexnet_final_preds = preds_tf_alex

# Build Transformer 6-class prob matrix
transformer_final_probs_ens = probs_transformer              # shape (N, 6) directly
transformer_final_preds_ens = preds_transformer

# NOTE: xgb_final_probs, lgbm_final_probs, deep_nn_final_probs,
#       xgb_final_preds, lgbm_final_preds, deep_nn_final_preds,
#       nn_final_preds are all already defined from their respective cells.

# Build shallow NN 6-class prob matrix (it only saved preds, not probs → reconstruct one-hot style)
# We use a hard one-hot so it contributes equally in soft vote
nn_final_probs = np.zeros((len(y_test), 6))
nn_final_probs[np.arange(len(y_test)), nn_final_preds] = 1.0

model_registry = [
    ("AlexNet",            alexnet_final_preds,      alexnet_final_probs),
    ("SVM + Deep Res MLP", deep_nn_final_preds,      deep_nn_final_probs),
    ("SVM + Shallow NN",   nn_final_preds,            nn_final_probs),
]

model_names = [m[0] for m in model_registry]
all_preds   = [m[1] for m in model_registry]
all_probs   = [m[2] for m in model_registry]
n_models    = len(model_registry)

# ─────────────────────────────────────────────────────
# 1. INDIVIDUAL MODEL ACCURACIES
# ─────────────────────────────────────────────────────
print("\n📊 Individual Model Accuracies:")
print("-" * 40)
for name, preds, _ in model_registry:
    acc = accuracy_score(y_test, preds)
    print(f"  {name:<25} {acc:.4f}")

# ─────────────────────────────────────────────────────
# 2. HARD VOTING (Majority Rule across all 5 models)
# ─────────────────────────────────────────────────────
stacked_preds = np.vstack(all_preds)                         # (n_models, N)
hard_vote_preds, _ = stats.mode(stacked_preds, axis=0)
hard_vote_preds = hard_vote_preds.flatten().astype(int)

print("\n\n--- 🗳️  Ensemble: HARD VOTING Metrics ---")
print(classification_report(y_test, hard_vote_preds, target_names=target_names))
plot_cm(y_test, hard_vote_preds, "Hard Voting Ensemble", target_names)

# ─────────────────────────────────────────────────────
# 3. SOFT VOTING (Average probabilities across all 5 models)
# ─────────────────────────────────────────────────────
soft_vote_probs = np.mean(np.stack(all_probs, axis=0), axis=0)   # (N, 6)
soft_vote_preds = np.argmax(soft_vote_probs, axis=1)

print("\n--- 🧮  Ensemble: SOFT VOTING Metrics ---")
print(classification_report(y_test, soft_vote_preds, target_names=target_names))
plot_cm(y_test, soft_vote_preds, "Soft Voting Ensemble", target_names)

# ─────────────────────────────────────────────────────
# 4. AGREEMENT BREAKDOWN (Hard Vote)
# ─────────────────────────────────────────────────────
print("\n🔬 AGREEMENT BREAKDOWN (Hard Vote)\n" + "=" * 50)

# Per-model correctness mask
correct_masks = [(preds == y_test) for preds in all_preds]

# Count how many models agreed on the hard vote winner per sample
agreement_counts = np.sum(
    np.vstack(all_preds) == hard_vote_preds[np.newaxis, :], axis=0
)  # shape (N,)

for threshold in [n_models, n_models - 1, n_models - 2]:
    mask = agreement_counts >= threshold
    n = np.sum(mask)
    if n == 0:
        continue
    acc = accuracy_score(y_test[mask], hard_vote_preds[mask])
    label = f"All {n_models}" if threshold == n_models else f"≥{threshold}/{n_models}"
    print(f"  {label} models agreed : {n:5d} samples  →  Hard-vote acc: {acc:.4f}")

# ─────────────────────────────────────────────────────
# 5. WRONG-DECISION CONFLICT MATRIX (Hard Vote)
#    Cell [i, j] = samples where model i was WRONG
#                  AND the hard vote final answer was WRONG
#                  AND model j was RIGHT
#    → "model j could have saved the vote but was outvoted by others incl. model i"
# ─────────────────────────────────────────────────────
print("\n🔴 WRONG-DECISION CONFLICT MATRIX (Hard Vote)\n" + "=" * 50)

hard_vote_wrong = (hard_vote_preds != y_test)          # samples where ensemble was wrong

conflict_matrix = np.zeros((n_models, n_models), dtype=int)

for i, (_, preds_i, _) in enumerate(model_registry):
    for j, (_, preds_j, _) in enumerate(model_registry):
        if i == j:
            continue
        # Hard vote was wrong, model i was also wrong, but model j was right
        condition = hard_vote_wrong & (preds_i != y_test) & (preds_j == y_test)
        conflict_matrix[i, j] = np.sum(condition)

conflict_df = pd.DataFrame(conflict_matrix, index=model_names, columns=model_names)

fig, ax = plt.subplots(figsize=(9, 6))
mask_diag = np.eye(n_models, dtype=bool)

sns.heatmap(
    conflict_df,
    annot=True,
    fmt='d',
    cmap='Reds',
    mask=mask_diag,
    linewidths=0.5,
    linecolor='#cccccc',
    ax=ax,
    cbar_kws={'label': 'Samples where ensemble voted wrong'}
)

ax.set_title(
    "Wrong-Decision Conflict Matrix (Hard Vote)\n"
    "Row = model was WRONG  ·  Column = model was RIGHT but outvoted",
    fontweight='bold', pad=14
)
ax.set_ylabel("Model that voted WRONG", labelpad=10)
ax.set_xlabel("Model that voted RIGHT (but was outvoted)", labelpad=10)
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
plt.tight_layout()
plt.show()

# ─────────────────────────────────────────────────────
# 6. INFERENCE TIME SUMMARY
# ─────────────────────────────────────────────────────
print("\n⏱️  INFERENCE TIME SUMMARY (ms / sample)")
print("-" * 45)
time_map = {
    "AlexNet":            tf_alex_time_ms,
    "SVM + Deep Res MLP": deep_nn_time_ms,
    "SVM + Shallow NN":   nn_time_ms,
    "SVM + XGBoost":      xgb_pipeline_time_ms,
}
for name, t in time_map.items():
    print(f"  {name:<25} {t:.4f} ms")
print(f"\n  {'Ensemble (bounded by slowest)':<25} ~{max(time_map.values()):.4f} ms")

In [ ]:
# import scipy.stats as stats

# print("🤝 CELL 6: ENSEMBLE & CONFLICT ANALYSIS\n" + "="*50)

# # 1. HARD VOTING (Majority Rule)
# stacked_preds = np.vstack([xgb_final_preds, lgbm_final_preds, tf_final_preds])
# hard_vote_preds, _ = stats.mode(stacked_preds, axis=0)
# hard_vote_preds = hard_vote_preds.flatten()

# # 2. SOFT VOTING (Averaging Probabilities)
# soft_vote_probs = (xgb_final_probs + lgbm_final_probs + tf_final_probs) / 3.0
# soft_vote_preds = np.argmax(soft_vote_probs, axis=1)

# print("\n--- Ensemble: Hard Voting Metrics ---")
# print(classification_report(y_test, hard_vote_preds, target_names=target_names))

# print("\n--- Ensemble: Soft Voting Metrics ---")
# print(classification_report(y_test, soft_vote_preds, target_names=target_names))
# plot_cm(y_test, soft_vote_preds, "Soft Voting Ensemble Classification", target_names)

# # 3. CONFLICT MATRIX (Agreement vs Correctness Analysis)
# print("\n🔬 CONFLICT MATRIX ANALYSIS\n" + "="*50)
# # We will identify how many models agreed and whether they were correct.

# agree_all = (xgb_final_preds == lgbm_final_preds) & (lgbm_final_preds == tf_final_preds)
# agree_two = ((xgb_final_preds == lgbm_final_preds) | 
#              (xgb_final_preds == tf_final_preds) | 
#              (lgbm_final_preds == tf_final_preds)) & ~agree_all
# agree_none = ~(agree_all | agree_two)

# correct_soft = (soft_vote_preds == y_test)

# print(f"Total Test Samples: {len(y_test)}")
# print(f"✅ All 3 Models Agreed: {np.sum(agree_all)} samples (Acc: {accuracy_score(y_test[agree_all], soft_vote_preds[agree_all]):.4f})")
# print(f"⚠️ Exactly 2 Models Agreed: {np.sum(agree_two)} samples (Acc: {accuracy_score(y_test[agree_two], soft_vote_preds[agree_two]):.4f})")
# print(f"❌ No Models Agreed: {np.sum(agree_none)} samples")

# # Pairwise Conflict Matrix: Rows = Model A Wrong, Columns = Model B Right
# models = {'XGB': xgb_final_preds, 'LGBM': lgbm_final_preds, 'ResNet': tf_final_preds}
# conflict_matrix = pd.DataFrame(0, index=models.keys(), columns=models.keys())

# for m1_name, m1_preds in models.items():
#     for m2_name, m2_preds in models.items():
#         if m1_name == m2_name:
#             continue
#         # Count cases where Model 1 was WRONG, but Model 2 was RIGHT
#         condition = (m1_preds != y_test) & (m2_preds == y_test)
#         conflict_matrix.loc[m1_name, m2_name] = np.sum(condition)

# plt.figure(figsize=(7, 5))
# sns.heatmap(conflict_matrix, annot=True, fmt='d', cmap='Reds', cbar_kws={'label': 'Number of Samples'})
# plt.title("Conflict Matrix\n(Row = Model was WRONG, Column = Model was RIGHT)", fontweight='bold')
# plt.ylabel('Model that Failed')
# plt.xlabel('Model that Succeeded')
# plt.show()

# # Inference Time Summary
# print("\n⏱️ INFERENCE TIME SUMMARY (ms/sample)")
# print("-" * 40)
# print(f"SVM + XGBoost Pipeline : {xgb_pipeline_time_ms:.4f} ms")
# print(f"SVM + LightGBM Pipeline: {lgbm_pipeline_time_ms:.4f} ms")
# print(f"1D ResNet (TensorFlow) : {tf_time_ms:.4f} ms")
# print(f"Ensemble (Parallel)    : ~{max(xgb_pipeline_time_ms, lgbm_pipeline_time_ms, tf_time_ms):.4f} ms (Bounded by slowest)")

In [ ]:
# sdvsdc